In [24]:
import os
import sys
import json
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd()
while not (project_root / 'requirements.txt').exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root (looked for requirements.txt)")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.models.classification import ClassificationModelHandler
from src.utils.training import test_model, training_loop

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

Using: cuda


In [26]:
DATASET = "BUSUCLM"
FEATURE_GROUP = "concatenated"
FEATURE_VARIANT = "basic_resnet" #basic_resnet, basic_shallow, trainable_resnet, trainable_shallow

num_classes = 2
batch_size = 32
num_epochs = 500
learning_rate = 0.001

offline_features_dir = project_root / 'experiments' / 'models' / 'fusion' / 'offline' / DATASET / FEATURE_GROUP / FEATURE_VARIANT
labels_dir = project_root / 'experiments' / 'labels' / DATASET
save_dir = project_root / 'experiments' / 'models' / 'classification' / 'offline' / DATASET / FEATURE_VARIANT
results_dir = project_root / 'experiments' / 'results' / 'offline' / DATASET / FEATURE_VARIANT

train_features_file = offline_features_dir / 'train_features_concat.npy'
val_features_file = offline_features_dir / 'val_features_concat.npy'
test_features_file = offline_features_dir / 'test_features_concat.npy'

train_labels_file = labels_dir / 'train_labels.npy'
val_labels_file = labels_dir / 'val_labels.npy'
test_labels_file = labels_dir / 'test_labels.npy'

model_name = f"classification_{DATASET.lower()}_{FEATURE_VARIANT}"

os.makedirs(save_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print(f"Dataset: {DATASET}")
print(f"Feature group: {FEATURE_GROUP}")
print(f"Feature variant: {FEATURE_VARIANT}")
print(f"Features directory: {offline_features_dir}")
print(f"Labels directory: {labels_dir}")
print(f"Model directory: {save_dir}")
print(f"Results directory: {results_dir}")

Dataset: BUSUCLM
Feature group: concatenated
Feature variant: basic_resnet
Features directory: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/fusion/offline/BUSUCLM/concatenated/basic_resnet
Labels directory: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/labels/BUSUCLM
Model directory: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/classification/offline/BUSUCLM/basic_resnet
Results directory: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/results/offline/BUSUCLM/basic_resnet


In [27]:
train_features = np.load(train_features_file)
val_features = np.load(val_features_file)
test_features = np.load(test_features_file)

train_labels = np.load(train_labels_file).squeeze().astype(np.int64)
val_labels = np.load(val_labels_file).squeeze().astype(np.int64)
test_labels = np.load(test_labels_file).squeeze().astype(np.int64)

print(f"Train features shape: {train_features.shape}")
print(f"Validation features shape: {val_features.shape}")
print(f"Test features shape: {test_features.shape}")
print(f"Train labels shape: {train_labels.shape}")
print(f"Validation labels shape: {val_labels.shape}")
print(f"Test labels shape: {test_labels.shape}")
print(f"Feature dimension: {train_features.shape[1]}")

Train features shape: (546, 256)
Validation features shape: (78, 256)
Test features shape: (156, 256)
Train labels shape: (546,)
Validation labels shape: (78,)
Test labels shape: (156,)
Feature dimension: 256


In [28]:
def build_loader(features, labels, batch_size, shuffle=False):
    features_tensor = torch.FloatTensor(features)
    labels_tensor = torch.LongTensor(labels)
    dataset = TensorDataset(features_tensor, labels_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

val_loader = build_loader(val_features, val_labels, batch_size=batch_size, shuffle=False)
test_loader = build_loader(test_features, test_labels, batch_size=batch_size, shuffle=False)

print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

Number of validation batches: 3
Number of test batches: 5


In [29]:
train_loader = build_loader(train_features, train_labels, batch_size=batch_size, shuffle=True)

print(f"Number of training batches: {len(train_loader)}")

Number of training batches: 18


In [30]:
sequential_hyperparams = {
    'input_dim': train_features.shape[1],  
    'learning_rate': learning_rate
}

classifier = ClassificationModelHandler(
    device=device,
    num_classes=num_classes,
    sequential_hyperparams=sequential_hyperparams
)

print("Model:")
print(classifier)

Model:


In [31]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.model.parameters(), lr=learning_rate)

history = training_loop(
    model=classifier.model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    save_dir=str(save_dir),
    model_name=model_name,
    extract_features=False,
    save_best_only=True,
    save_all_epochs=False
)

best_model_path = save_dir / f"best_{model_name}.pth"

print("\nTraining completed.")
print(f"Best model path: {best_model_path}")

Epoch 1/500



Epoch 1 Summary:
  Train - Loss: 0.7617, Acc: 0.5824
  Val   - Loss: 0.8063, Acc: 0.4615, Precision: 0.7143, Recall: 0.4386, F1: 0.5435, AUC: 0.5230
  ✓ New best model saved (val_acc: 0.4615) at /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/classification/offline/BUSUCLM/basic_resnet/best_classification_busuclm_basic_resnet.pth
Epoch 2/500



Epoch 2 Summary:
  Train - Loss: 0.7157, Acc: 0.5824
  Val   - Loss: 0.7064, Acc: 0.4744, Precision: 0.7222, Recall: 0.4561, F1: 0.5591, AUC: 0.5146


  ✓ New best model saved (val_acc: 0.4744) at /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/classification/offline/BUSUCLM/basic_resnet/best_classification_busuclm_basic_resnet.pth
Epoch 3/500



Epoch 3 Summary:
  Train - Loss: 0.6811, Acc: 0.6007
  Val   - Loss: 0.7633, Acc: 0.4231, Precision: 0.6875, Recall: 0.3860, F1: 0.4944, AUC: 0.5188
Epoch 4/500


...


Epoch 496 Summary:
  Train - Loss: 0.5910, Acc: 0.6703
  Val   - Loss: 0.7829, Acc: 0.5769, Precision: 0.7222, Recall: 0.6842, F1: 0.7027, AUC: 0.4845
Epoch 497/500



Epoch 497 Summary:
  Train - Loss: 0.5458, Acc: 0.6832
  Val   - Loss: 0.8506, Acc: 0.5128, Precision: 0.7021, Recall: 0.5789, F1: 0.6346, AUC: 0.4845
Epoch 498/500



Epoch 498 Summary:
  Train - Loss: 0.5521, Acc: 0.6777
  Val   - Loss: 0.7750, Acc: 0.5769, Precision: 0.7143, Recall: 0.7018, F1: 0.7080, AUC: 0.4536
Epoch 499/500



Epoch 499 Summary:
  Train - Loss: 0.5539, Acc: 0.6630
  Val   - Loss: 0.8820, Acc: 0.4744, Precision: 0.6818, Recall: 0.5263, F1: 0.5941, AUC: 0.4662
Epoch 500/500



Epoch 500 Summary:
  Train - Loss: 0.5708, Acc: 0.6667
  Val   - Loss: 0.8140, Acc: 0.5641, Precision: 0.7347, Recall: 0.6316, F1: 0.6792, AUC: 0.5125

Training completed.
Best model path: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/classification/offline/BUSUCLM/basic_resnet/best_classification_busuclm_basic_resnet.pth


In [32]:
best_model_path = save_dir / f"best_{model_name}.pth"

if best_model_path.exists():
    classifier.model.load_state_dict(torch.load(best_model_path, map_location=device))
    print(f"Best model loaded from: {best_model_path}")
else:
    print("Warning: best model file not found. Using current model weights.")

classifier.model.eval()

/tmp/ipykernel_318844/2620730251.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  classifier.model.load_state_dict(torch.load(best_model_path, map_location=device))


Best model loaded from: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/models/classification/offline/BUSUCLM/basic_resnet/best_classification_busuclm_basic_resnet.pth


Sequential(
  (0): Linear(in_features=256, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=128, out_features=2, bias=True)
)

In [33]:
criterion = nn.CrossEntropyLoss()

hyperparams = {
    'dataset': DATASET,
    'feature_group': FEATURE_GROUP,
    'feature_variant': FEATURE_VARIANT,
    'num_classes': num_classes,
    'batch_size': batch_size,
    'num_epochs': num_epochs,
    'learning_rate': learning_rate,
    'input_dim': train_features.shape[1]
}

output_file = results_dir / 'test_results.txt'
metrics_file = results_dir / 'metrics_summary.json'

test_loss, accuracy, precision, recall, f1, auc = test_model(
    model=classifier.model,
    dataloader=test_loader,
    criterion=criterion,
    device=device,
    hyperparams=hyperparams,
    output_file=str(output_file),
    extract_features=False,
    compute_detailed_metrics=True
)

summary = {
    'dataset': DATASET,
    'feature_group': FEATURE_GROUP,
    'feature_variant': FEATURE_VARIANT,
    'model_name': model_name,
    'best_model_path': str(best_model_path),
    'results_file': str(output_file),
    'metrics': {
        'test_loss': float(test_loss),
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'auc': float(auc)
    }
}

with open(metrics_file, 'w') as f:
    json.dump(summary, f, indent=4)

print(f"\n{'='*50}")
print("FINAL TEST RESULTS:")
print(f"{'='*50}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"{'='*50}")
print(f"Detailed results saved at: {output_file}")
print(f"Metrics summary saved at: {metrics_file}")

Testing: 100%|██████████| 5/5 [00:00<00:00, 1085.82it/s]


Test Loss: 0.6922
Accuracy: 0.6090, Precision: 0.7265, Recall: 0.7456, F1: 0.7359, AUC: 0.5379
Macro - Precision: 0.4915, Recall: 0.4919, F1: 0.4914
Micro - Precision: 0.6090, Recall: 0.6090, F1: 0.6090

FINAL TEST RESULTS:
Test Loss: 0.6922
Accuracy: 0.6090
Precision: 0.7265
Recall: 0.7456
F1-Score: 0.7359
AUC: 0.5379
Detailed results saved at: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/results/offline/BUSUCLM/basic_resnet/test_results.txt
Metrics summary saved at: /home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/results/offline/BUSUCLM/basic_resnet/metrics_summary.json
